In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
path = 'resources'

In [4]:
#Get a list of all csv file paths in the folder
all_files = glob.glob(os.path.join(path, "*.csv"))

#Create a list of dataframes
df_list = []

for filename in all_files:
    df = pd.read_csv(filename)
    df_list.append(df)

#Find the intersection of columns across all DataFrames, preserving order from the first DataFrame
if df_list:
    #Get columns from the first DataFrame as a reference for order
    first_df_cols = df_list[0].columns.tolist()
    all_df_cols_sets = [set(df.columns) for df in df_list]
    common_cols_set = set.intersection(*all_df_cols_sets)
    common_cols = [col for col in first_df_cols if col in common_cols_set]
else:
    common_cols = []

#Select only the common columns from each DataFrame, in the preserved order
df_list_common = [df[common_cols] for df in df_list]

#Stack them vertically
weather_df = pd.concat(df_list_common, axis=0, ignore_index=True)

#Check the results
print(f"Combined {len(all_files)} files.")
print(f"Number of common columns kept: {len(common_cols)}")
weather_df.head()

EmptyDataError: No columns to parse from file

In [ ]:
#Identify all columns that start with "WT" (weather type binaries)
wt_cols = [col for col in weather_df.columns if col.startswith('WT')]

#Fill the missing (NaN) values with 0
weather_df[wt_cols] = weather_df[wt_cols].fillna(0).astype(int)

#Verify the change
print(f"Updated {len(wt_cols)} columns: {wt_cols}")
weather_df[wt_cols].head()

In [ ]:
#Rename weather types to reflect their meanings
weather_map = {
    'WT01': 'Fog_Ice_Fog',
    'WT02': 'Heavy_Fog',
    'WT03': 'Thunder',
    'WT04': 'Ice_Pellets_Sleet',
    'WT05': 'Hail',
    'WT06': 'Glaze_Rime',
    'WT07': 'Dust_Ash_Sand',
    'WT08': 'Smoke_Haze',
    'WT09': 'Blowing_Snow',
    'WT10': 'Tornado_Funnel_Cloud',
    'WT11': 'High_Winds',
    'WT12': 'Blowing_Spray',
    'WT13': 'Mist',
    'WT14': 'Drizzle',
    'WT15': 'Freezing_Drizzle',
    'WT16': 'Rain',
    'WT17': 'Freezing_Rain',
    'WT18': 'Snow',
    'WT19': 'Unknown_Precip',
    'WT21': 'Ground_Fog',
    'WT22': 'Ice_Fog'
}

#Apply the rename
weather_df = weather_df.rename(columns=weather_map)
weather_df.head()

In [ ]:
#Convert data types
weather_df['DATE'] = pd.to_datetime(weather_df['DATE'])

In [ ]:
#Aggregate so there is only one row per date, preferring O'Hare as the station whenever available (because it's most consistently available in the dataset)
weather_df['preference'] = np.where(weather_df['NAME'] == 'CHICAGO OHARE INTERNATIONAL AIRPORT, IL US', 0, 1)
weather = weather_df.sort_values(by=['DATE', 'preference']).drop_duplicates(subset=['DATE'], keep='first').drop(columns=['preference'])
display(weather)

In [ ]:
output_path = ('resources')
file_name = 'weather_data'

#Save as .pkl
weather.to_pickle(os.path.join(output_path, f'{file_name}.pkl'))
print(f"DataFrame saved as {file_name}.pkl in {output_path}")

#Save as .csv
weather.to_csv(os.path.join(output_path, f'{file_name}.csv'), index=False)
print(f"DataFrame saved as {file_name}.csv in {output_path}")